In [ ]:
# -*- coding: utf-8 -*-
"""
Batch post-processing for radiomics CSV files exported from 3D Slicer.

What this script does:
- Reads all CSV files from a source folder
- For each CSV file (sorted by filename):
    * Extracts the 4th column
    * Starts from row 39 to the end (1-based indexing)
    * Converts that vertical column into one horizontal row
- Writes the extracted values into a target CSV file

Output layout:
- The first source CSV is written to row 2
- The second source CSV is written to row 3
- ...
- Writing always starts from column 3 in the target file

Typical use:
This is useful for reorganizing 3D Slicer radiomics output into a
sample-by-feature table structure that is easier to use in machine
learning workflows.

Notes:
- Filenames are not written into the output table
- The target CSV will be created if it does not exist
- The file is saved as UTF-8 with BOM for better Excel compatibility
"""

import csv
from pathlib import Path

# Example input folder containing radiomics CSV files exported from 3D Slicer
CONTROL_DIR = Path(r"D:\ExampleProject\RadiomicsData\Control_CSVs")

# Example output file used to store the reorganized feature table
RADIOMICS_CSV = Path(r"D:\ExampleProject\RadiomicsData\Radiomics.csv")

# Source data settings
START_ROW_1B = 39   # Start reading from row 39 (inclusive), using 1-based indexing
SRC_COL_1B   = 4    # Read the 4th column

# Destination layout settings
DEST_START_ROW_1B = 2  # The first extracted sample will be written to row 2
DEST_START_COL_1B = 3  # Writing starts from column 3 in each target row

# Try these encodings when opening source CSV files
READ_ENCODINGS = ["utf-8-sig", "utf-8", "gbk"]


def list_csvs(folder: Path):
    """Return all CSV files in the folder, sorted by filename."""
    if not folder.exists() or not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == '.csv'],
        key=lambda p: p.name.lower()
    )


def read_col(csv_path: Path, col_1b: int, start_row_1b: int):
    """
    Read one column from a CSV file, starting at a given row.

    Parameters
    ----------
    csv_path : Path
        Path to the CSV file
    col_1b : int
        Column number in 1-based indexing
    start_row_1b : int
        Starting row in 1-based indexing

    Returns
    -------
    list[str]
        Values from the selected column, from the start row to the end
    """
    rows = None

    # Try several common encodings in case the CSV file is not UTF-8
    for enc in READ_ENCODINGS:
        try:
            with csv_path.open('r', encoding=enc, newline='') as f:
                rows = list(csv.reader(f))
            break
        except Exception:
            rows = None

    if rows is None:
        raise RuntimeError(f"Failed to read CSV: {csv_path}")

    col_idx = max(0, col_1b - 1)
    start_idx = max(0, start_row_1b - 1)

    out = []
    for r in rows[start_idx:]:
        # If the row is shorter than expected, write an empty string instead
        out.append(r[col_idx].strip() if len(r) > col_idx else "")

    return out


def ensure_rows(mat, min_rows):
    """Expand the table to make sure it has at least the required number of rows."""
    while len(mat) < min_rows:
        mat.append([])


def ensure_len(row, ncols):
    """Expand one row to make sure it has at least the required number of columns."""
    if len(row) < ncols:
        row.extend([""] * (ncols - len(row)))


def main():
    # Get all source CSV files
    files = list_csvs(CONTROL_DIR)

    # Load the target CSV if it already exists; otherwise start with an empty table
    table = []
    if RADIOMICS_CSV.exists():
        with RADIOMICS_CSV.open('r', encoding='utf-8-sig', newline='') as f:
            table = list(csv.reader(f))

    if not table:
        table = []

    # Process each source CSV file one by one
    for i, fp in enumerate(files):
        # Read the target column as a vertical list
        values = read_col(fp, SRC_COL_1B, START_ROW_1B)

        # Decide which row in the destination table to write into
        dest_row_1b = DEST_START_ROW_1B + i

        # Make sure the destination table is large enough
        ensure_rows(table, dest_row_1b)
        row = table[dest_row_1b - 1]

        # Writing starts from the specified destination column
        start_idx = max(0, DEST_START_COL_1B - 1)
        ensure_len(row, start_idx + len(values))

        # Write the extracted values horizontally across the row
        for j, v in enumerate(values):
            row[start_idx + j] = v

        table[dest_row_1b - 1] = row

    # Save the updated table
    with RADIOMICS_CSV.open('w', encoding='utf-8-sig', newline='') as f:
        csv.writer(f).writerows(table)

    print(f"[OK] Processed {len(files)} CSV files from {CONTROL_DIR}.")
    print(f"  Wrote each sample to Radiomics.csv starting at row {DEST_START_ROW_1B}, column {DEST_START_COL_1B}.")


if __name__ == "__main__":
    main()